In [13]:
import pandas as pd
import random
import spacy
from tqdm import tqdm
import numpy as np

nlp = spacy.load("de_core_news_sm")

# df = pd.read_csv('data/news_data.csv')
df = pd.read_csv('data/news_data_text_5000.csv', sep=';')

In [14]:
import re

# df_title = df["title"].dropna()
# df_text = df["text"].dropna()
df_de = df["text"].dropna()

# df_de = pd.concat([df_title, df_text])

def clean(text: str) -> str:
    # remove: „ “ """ 
    text = re.sub(r'[„“"""\n]', '', text)

    # Clean up excessive punctuation (keep up to 3 repetitions)
    text = re.sub(r'([!?.]){4,}', r'\1\1\1', text)

    # Remove extra whitespace
    text = ' '.join(text.split())

    return text

# clean("Folgen der Räumung? – „ \n\nDa aknn man sihc die ganze Bandbreite vosrtellen “")

len(df_de)

77827

# Corruption Strategy 

- Use spaCy (de_core_news_sm) to extract POS and morphology.
- Write 5–10 custom error rule functions:
  - Gender/article confusion
  - Verb conjugation error
  - Case (nominative/accusative) errors
  -  Preposition omission
  -  Word order change
  -  Random word deletion
  - Typos (via nlpaug.KeyboardAug)
- apply 1–3 random transformations per sentence.
- Mix the result with clean text.

# Gemini

- max 1-2 error per sentence
- Start Simple: Start by training on 50% "Case/Article" errors and 50% "Typo/Capitalization" errors. These are the highest frequency errors in real life.
- Keep Some "Clean" Data: Include examples where incorrect == correct. The model needs to learn that sometimes a sentence is already perfect and shouldn't be changed. (This prevents "over-correction"

1. Mechanical Errors (Easy & High Volume)

- also introduce lowercasing the whole sentence
- De-Capitalization (Noun Lowercasing) - Der tisch ist groß.
- Umlaut Flattening: Map: ä→a, ö→o, ü→u, ß→ss - Ich bin mude.
- Homophone Confusion: Pairs: das/dass, seid/seit, war/wahr, wieder/wider - Ich weiß, das du kommst.

2. Morphological Errors (Medium Difficulty)

- Article Confusion - Ich gebe den Mann die Buch.
- Preposition Swaps - Ich warte an dich.
- Adjective Endings - Ein schnelle Auto.

3. Syntax Errors (Harder to Implement)

- Verb Position - Heute ich gehe ins Kino.
- Compound Word Splitting - Donaudampf schifffahrt

In [15]:
def list_replacer(sentence: str, replacement_list: list[str]) -> str:
    doc = nlp(sentence)
    tokens = [t.text for t in doc]
    # print(tokens)

    for i, token in enumerate(doc):
        if token.text.lower() in replacement_list:
            replacements = [art for art in replacement_list if art != token.text.lower()]
            selected = random.choice(replacements)
            if token.text.islower():
                selected = selected.lower()
            elif token.text.istitle():
                selected = selected.title()
            else:
                selected = selected.upper()
            tokens[i] = selected

    return " ".join(tokens).replace(" .", ".").replace(" !", "!").replace(" ?", "?").replace(" ,", ",")

# Article confusion
list_replacer("Der Hund spielt mit dem Ball und die Katze schläft auf dem Sofa.", replacement_list=["der", "die", "das", "den", "dem", "des"])
# list_replacer("Ein Junge spielt mit einem Ball. ", replacement_list=["ein", "eine", "einen", "einem", "einer", "eines"])
# list_replacer("Ein Junge spielt mit einem Ball. ", replacement_list=["kein", "keine", "keinen", "keinem", "keiner", "keines"])
# list_replacer("Ein Junge spielt mit einem Ball und mir. ", replacement_list=["ich", "du", "er", "sie", "es", "wir", "ihr", "sie", "mich", "dich", "ihn", "uns", "euch", "ihnen", "mir", "dir", "ihm"])

'Den Hund spielt mit das Ball und dem Katze schläft auf des Sofa.'

In [16]:
def introduce_conjugation_error(sentence):
    doc = nlp(sentence)
    tokens = [t.text for t in doc]
    # print(tokens)

    for i, token in enumerate(doc):
        # print(token.text, token.pos_, token.morph)
        # Identify finite verbs (best target for conjugation errors)
        if token.pos_ in ["VERB", "AUX"] and token.morph.get("VerbForm") == ["Fin"]:

            original = token.text
            lemma = token.lemma_

            # Simple error strategies:
            # 1) Replace with infinitive
            if random.random() < 0.4:
                tokens[i] = lemma  # infinitive form
                continue

            # 2) Replace with incorrect suffix
            if original.endswith("e") and random.random() < 0.5:
                tokens[i] = original[:-1] + "t"
                continue

            if original.endswith("t") and random.random() < 0.5:
                tokens[i] = original[:-1] + "en"
                continue

            # 3) Add wrong person ending
            wrong_endings = ["e", "st", "t", "en"]
            wrong = random.choice(wrong_endings)
            tokens[i] = lemma + wrong

    # Rebuild corrupted sentence
    return " ".join(tokens).replace(" .", ".").replace(" !", "!").replace(" ?", "?").replace(" ,", ",")

introduce_conjugation_error("Der Hund spielt mit dem roten Ball und die Katze schläft auf dem Sofa.")

'Der Hund spielenst mit dem roten Ball und die Katze schlafen auf dem Sofa.'

In [17]:
ADJ_ENDINGS = ["", "e", "en", "er", "es", "em"]

def introduce_adjective_error(sentence):
    doc = nlp(sentence)
    tokens = []

    for token in doc:
        if token.pos_ == "ADJ":
            # Choose the error type
            error_type = random.choice([
                "wrong_ending",
                "strip_ending",
            ])
            
            if error_type == "wrong_ending":
                base = token.text.rstrip("eernsm")  # rough stem
                new_ending = random.choice(ADJ_ENDINGS)
                corrupted = base + new_ending
                tokens.append(corrupted)

            elif error_type == "strip_ending":
                base = token.text.rstrip("eernsm")
                tokens.append(base)
                
        else:
            tokens.append(token.text)


    return " ".join(tokens).replace(" .", ".").replace(" !", "!").replace(" ?", "?").replace(" ,", ",")

# Example
sent = "Ich sehe einen großen Hund im schönen Park."
for _ in range(5):
    print(introduce_adjective_error(sent))


Ich sehe einen groß Hund im schöen Park.
Ich sehe einen großes Hund im schö Park.
Ich sehe einen großer Hund im schö Park.
Ich sehe einen groß Hund im schöer Park.
Ich sehe einen großem Hund im schö Park.


In [18]:
def preposition_omission(sentence: str) -> str:
    doc = nlp(sentence)
    tokens = [t.text for t in doc]
    # print(tokens)

    for i, token in enumerate(doc):
        if token.pos_ == "ADP":  # Adposition (preposition/postposition)
            tokens[i] = ""

    return " ".join(tokens).replace("  ", " ").replace(" .", ".").replace(" !", "!").replace(" ?", "?").replace(" ,", ",")

preposition_omission("Der Hund spielt mit dem Ball und die Katze schläft auf dem Sofa.")

'Der Hund spielt dem Ball und die Katze schläft dem Sofa.'

In [19]:
def random_word_deletion(sentence: str, deletion_prob: float = 0.2) -> str:
    doc = nlp(sentence)
    tokens = []

    if len(doc) < 3:
        return sentence  # Avoid deleting too much in short sentences

    for token in doc:
        if random.random() > deletion_prob:
            tokens.append(token.text)

    return " ".join(tokens).replace("  ", " ").replace(" .", ".").replace(" !", "!").replace(" ?", "?").replace(" ,", ",")

random_word_deletion("Der Hund spielt mit dem Ball und die Katze schläft auf dem Sofa.", deletion_prob=0.2)

'Der spielt mit dem Ball und Katze auf dem Sofa.'

In [20]:
def random_typos(sentence: str, typo_prob: float = 0.2) -> str:
    doc = nlp(sentence)
    tokens = []

    for token in doc:
        word = token.text
        if random.random() < typo_prob and len(word) > 1:
            char_list = list(word)
            idx1 = random.randint(0, len(char_list) - 2)
            idx2 = idx1 + 1
            # Swap characters
            char_list[idx1], char_list[idx2] = char_list[idx2], char_list[idx1]
            word = "".join(char_list)
        tokens.append(word)

    return " ".join(tokens).replace("  ", " ").replace(" .", ".").replace(" !", "!").replace(" ?", "?").replace(" ,", ",")

random_typos("Der Hund spielt mit dem Ball und die Katze schläft auf dem Sofa.", typo_prob=0.2)

'Dre Hund spielt mit dem Blal und dei Kazte schläft auf dem Sofa.'

In [21]:
def article_replacer(sentence: str) -> str:
    return list_replacer(sentence, replacement_list=["der", "die", "das", "den", "dem", "des"])

def tow_form_determinant_replacer(sentence: str) -> str:
    return list_replacer(sentence, replacement_list=["ein", "eine", "einen", "einem", "einer", "eines", "kein", "keine", "keinen", "keinem", "keiner", "keines"])

def pronoun_replacer(sentence: str) -> str:
    return list_replacer(sentence, replacement_list=["ich", "du", "er", "sie", "es", "wir", "ihr", "sie", "mich", "dich", "ihn", "uns", "euch", "ihnen", "mir", "dir", "ihm"])


corruptions = [
    article_replacer,
    tow_form_determinant_replacer,
    pronoun_replacer,
    introduce_conjugation_error,
    introduce_adjective_error,
    preposition_omission,
    random_word_deletion,
    random_typos
]

def apply_random_corruption(sentence: str, num_corruptions: int = 2) -> str:
    corrupted_sentence = sentence

    while corrupted_sentence == sentence:
        selected_corruptions = random.sample(corruptions, num_corruptions)

        for corruption in selected_corruptions:
            corrupted_sentence = corruption(corrupted_sentence)

    return corrupted_sentence

In [22]:
def process_chunk(df_chunk):
    tqdm.pandas()
    df_corrupted_chunk = df_chunk.progress_apply(lambda x: apply_random_corruption(x, num_corruptions=2))
    return df_corrupted_chunk


# for i, df_chunk in enumerate(np.array_split(df_de, 4)):
#     df_corrupted_chunk = process_chunk(df_chunk)
#     df_corrupted_chunk = pd.DataFrame({ 'de_correct': df_chunk, 'de_corrupted': df_corrupted_chunk })
#     df_corrupted_chunk.to_csv(f'data/news_data_corrupted_{i}.csv', sep=';', index=False)
    
#     count_same = sum(df_corrupted_chunk['de_correct'] == df_corrupted_chunk['de_corrupted'])
#     print(f"Number of unchanged sentences: {count_same} out of {len(df_corrupted_chunk)}")

# df_de_cleaned = df_de.apply(lambda x: clean(x))
df_corrupted_chunk = process_chunk(df_de[40000:])
df_corrupted_chunk = pd.DataFrame({ 'de_correct': df_de[40000:], 'de_corrupted': df_corrupted_chunk })
count_same = sum(df_corrupted_chunk['de_correct'] == df_corrupted_chunk['de_corrupted'])
print(f"Number of unchanged sentences: {count_same} out of {len(df_corrupted_chunk)}")

100%|██████████| 37827/37827 [10:53<00:00, 57.92it/s]

Number of unchanged sentences: 0 out of 37827


In [23]:
# df_corrupted = pd.DataFrame({ 'de_correct': df_de, 'de_corrupted': df_corrupted })

df_corrupted_chunk.to_csv('data/news_data_corrupted_5000_2.csv', sep=';', index=False)

In [24]:
# count_same = sum(df_corrupted['de_correct'] == df_corrupted['de_corrupted'])
# print(f"Number of unchanged sentences: {count_same} out of {len(df_corrupted)}")